Created Schema for medallion architecture

create schema bronze

create schema silver

create schema gold

Loading KARDS , SPAWNABLES and FORECAST data into bronze

In [1]:
table_names = ["KARDS","SPAWNABLES","FORECAST","EXILE","SYNERGY_TAG_RULES","SYNERGY_COMBO_RULES"]


for table in table_names:
    sql = f"DROP TABLE bronze.{table}"
    spark.sql(sql)
    file_string = f"{table}.parquet"
    df = spark.read.parquet(f"Files/{file_string}")
    df.write.format("delta").mode("overwrite").saveAsTable(f"bronze.{table}")

StatementMeta(, 7730de74-ff17-4d4a-bba4-23b39bffd747, 3, Finished, Available, Finished, False)

Transfer bronze raw data to silver layer

create table silver.kards as select * from bronze.kards

create table silver.spawnables as select * from bronze.spawnables

create table silver.forecast as select * from bronze.forecast

Transformation logic from bronze to silver

In [2]:
df = spark.table("bronze.kards")

df_silver = df.select(
    "CardId", "CardName", "CardType", "CardNation", "CardRarity",
    "CardSubType", "CostToPlay", "CostToOperate", "Attack", "HitPoint",
    "Keywords", "CardEffect", "IsVeteran", "VeteranCostToOperate",
    "VeteranAttack", "VeteranHitPoint", "VeteranKeywords", "VeteranEffect",
    "Status", "Expansion", "IsPermanentPool", "IsSpawnable", "IsForecastable"
)
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.kards")


df = spark.table("bronze.spawnables")

df_silver = df.drop(
    "SpawnImagePaths", "Spawn6KImagePaths", "Spawn9KImagePaths", "Spawn12KImagePaths","createddate","modifieddate"
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.spawnables")

df = spark.table("bronze.forecast")

df_silver = df.drop(
    "ForecastCardImagePath", "ParentCardImagePath","createddate","modifieddate"
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.forecast")

df = spark.table("bronze.EXILE")
df_silver = df.drop(
    "createddate","modifieddate"
)
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.EXILE")

df = spark.table("bronze.SYNERGY_TAG_RULES")
df_silver = df.drop(
    "createddate","modifieddate"
)
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.SYNERGY_TAG_RULES")

df = spark.table("bronze.SYNERGY_COMBO_RULES")
df_silver = df.drop(
    "createddate","modifieddate"
)
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.SYNERGY_COMBO_RULES")


StatementMeta(, 7730de74-ff17-4d4a-bba4-23b39bffd747, 4, Finished, Available, Finished, False)

Null check on kards bronze as it is the main table

In [3]:
from pyspark.sql.functions import col, sum as spark_sum, when

df = spark.table("bronze.KARDS")
null_counts = df.select([spark_sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
for c in df.columns])
display(null_counts) #nulls are legitimate and not 

StatementMeta(, 7730de74-ff17-4d4a-bba4-23b39bffd747, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 11849934-a084-4b11-8dda-a0a3b29bf4ae)

In [4]:
from pyspark.sql.functions import col

kards = spark.table("silver.kards").filter(col("IsSpawnable") == True) # spark.table(tab).filter(col(column) (some condition))  --filter returns bool value
spawnables = spark.table("silver.spawnables")

spawn_chain = kards.join(spawnables, kards.CardId == spawnables.CardId, "inner") #table1.join(table2 , join_condition , join_type)
spawn_chain = spawn_chain.drop(spawnables.CardId)
spawn_chain.write.format("delta").mode("overwrite").saveAsTable("gold.spawn_chain") #table.format("delta").mode("overwrite").saveAsTable(table_01)


eligible_for_forecast = spark.table("silver.kards").filter(col("IsForecastable") == True)

eligible_for_forecast.write.format("delta").mode("overwrite").saveAsTable("gold.eligible_for_forecast")

veteran_cards = spark.table("silver.kards").filter(col("IsVeteran") == True)

veteran_cards.write.format("delta").mode("overwrite").saveAsTable("gold.veteran_cards")

permanent_pool_cards = spark.table("silver.kards").filter(col("IsPermanentPool") == True)

permanent_pool_cards.write.format("delta").mode("overwrite").saveAsTable("gold.permanent_pool_cards")

k = spark.table("silver.kards")
e = spark.table("silver.exile")
exile_data = k.join(e, "CardId", "inner").select(k.CardId, k.CardName, k.CardType, k.CardNation, k.CardRarity,
    k.CardSubType, k.CostToPlay, k.CostToOperate, k.Attack, k.HitPoint,
    k.Keywords, k.CardEffect, k.IsVeteran, k.VeteranCostToOperate,
    k.VeteranAttack, k.VeteranHitPoint, k.VeteranKeywords, k.VeteranEffect,
    k.Status, k.Expansion, k.IsPermanentPool, k.IsSpawnable, k.IsForecastable,e.ExileCardId,e.AlsoUsedBy)
exile_data.write.format("delta").mode("overwrite").saveAsTable("gold.exile_data")

tables = ["kards","spawnables","forecast","exile","synergy_tag_rules","synergy_combo_rules"]

for table in tables:
    df = spark.table(f"silver.{table}")
    df.write.format("delta").mode("overwrite").saveAsTable(f"gold.{table}")

StatementMeta(, 7730de74-ff17-4d4a-bba4-23b39bffd747, 6, Finished, Available, Finished, False)